In [74]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [75]:
result_path = Path("../thesis_results/downstream_task")
methods_list = ["uniform","psa", "kmm", "mrs-forest", "soft-mrs-exponential"]
# bias_types = ["less_negative_class", "less_positive_class", "mean_difference"]
bias_types = ["mean_difference"]
metrics = ["MMDs",]
less_bias_strengths = ["0.1"]
mean_bias_strengthts = ["0.8"]
datasets = [("folktables_employment", "Employment_relative_bias"), ("folktables_income", "Binary Income_relative_bias"), 
            ("hr_analytics", "target_relative_bias"), 
            ("breast_cancer", "class_relative_bias"), ("loan_prediction", "Loan_Status_relative_bias")]

In [76]:
aurocs = []
auprcs = []
dict_list = []
for dataset, target in datasets:
    for bias_type in bias_types:
        for method in methods_list:
            if bias_type == "mean_difference":
                bias_strengths = mean_bias_strengthts
            else : 
                bias_strengths = less_bias_strengths
            for bias_strength in bias_strengths:
                json_file = result_path / dataset / bias_type /  bias_strength/ method / "similarity_results.json"
                result_file = pd.read_json(str(json_file))
                dict_list.append(
                    {
                        "Method": method, "Data Set": dataset, 
                        "MMD Mean": result_file["MMDs"]["mean"], 
                        "MMD Std": result_file["MMDs"]["sd"], 
                        "Relative Bias Mean": result_file[target]["bias mean"], 
                        "Relative Bias Std": result_file[target]["bias sd"], 
                        
                        "Bias Type": bias_type, "Bias Strength": bias_strength,
                    }
                                )
result_df = pd.DataFrame(data=dict_list)

In [77]:
result_df = result_df.replace({"uniform": "Uniform", "kmm": "KMM", "psa": "PSA", "mrs-forest": "MRS",
                               "soft-mrs-linear": "Soft-MRS-Linear", "soft-mrs-exponential": "Soft-MRS-Exponential",
                               "fw-mrs-temperature-svm": "FW-MRS-SVM", "fw-mrs-temperature": "FW-MRS-RF"})
result_df


,Method,Data Set,MMD Mean,MMD Std,Relative Bias Mean,Relative Bias Std,Bias Type,Bias Strength
0,Uniform,folktables_employment,0.079679,0.003960,14.809437,0.686364,mean_difference,0.8
1,PSA,folktables_employment,0.053899,0.004547,9.390903,2.172712,mean_difference,0.8
2,KMM,folktables_employment,0.043961,0.004521,5.192807,2.580808,mean_difference,0.8
3,MRS,folktables_employment,0.070772,0.003329,12.086740,1.517933,mean_difference,0.8
4,Soft-MRS-Exponential,folktables_employment,0.042129,0.003904,2.571135,1.550010,mean_difference,0.8
5,Uniform,folktables_income,0.088443,0.005744,11.947570,0.943977,mean_difference,0.8
6,PSA,folktables_income,0.067264,0.005413,8.414665,1.772051,mean_difference,0.8
7,KMM,folktables_income,0.062693,0.005487,6.085281,3.296049,mean_difference,0.8
8,MRS,folktables_income,0.082183,0.005062,9.852303,1.264574,mean_difference,0.8
9,Soft-MRS-Exponential,folktables_income,0.047603,0.005070,2.752369,1.829118,mean_difference,0.8


In [78]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            mean_auroc_values = []
            std_auroc_values = []
            for dataset, target in datasets:
                mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["MMD Mean"].iloc[0]
                mean_auroc_values.append(np.round(mean_auroc, 3))

                std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["MMD Std"].iloc[0]
                std_auroc_values.append(np.round(std_auroc, 3))

                

            print(f"\t& {method} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ & \\\\")
        print("\n")

mean_difference, 0.8
	& Uniform & $0.08\pm0.004$ & $0.088\pm0.006$ & $0.082\pm0.004$ & $0.17\pm0.017$ & $0.099\pm0.017$ & \\
	& PSA & $0.054\pm0.005$ & $0.067\pm0.005$ & $0.052\pm0.003$ & $0.091\pm0.018$ & $0.063\pm0.017$ & \\
	& KMM & $0.044\pm0.005$ & $0.063\pm0.005$ & $0.04\pm0.003$ & $0.069\pm0.011$ & $0.034\pm0.006$ & \\
	& MRS & $0.071\pm0.003$ & $0.082\pm0.005$ & $0.075\pm0.004$ & $0.161\pm0.018$ & $0.094\pm0.015$ & \\
	& Soft-MRS-Exponential & $0.042\pm0.004$ & $0.048\pm0.005$ & $0.036\pm0.003$ & $0.06\pm0.01$ & $0.037\pm0.006$ & \\




In [79]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            mean_auroc_values = []
            std_auroc_values = []
            for dataset, target in datasets:
                mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["Relative Bias Mean"].iloc[0]
                mean_auroc_values.append(np.round(mean_auroc, 3))

                std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["Relative Bias Std"].iloc[0]
                std_auroc_values.append(np.round(std_auroc, 3))

                

            print(f"\t& {method} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ & \\\\")
        print("\n")

mean_difference, 0.8
	& Uniform & $14.809\pm0.686$ & $11.948\pm0.944$ & $3.221\pm0.681$ & $24.045\pm0.92$ & $5.093\pm3.655$ & \\
	& PSA & $9.391\pm2.173$ & $8.415\pm1.772$ & $2.3\pm0.287$ & $6.293\pm2.621$ & $2.0\pm2.159$ & \\
	& KMM & $5.193\pm2.581$ & $6.085\pm3.296$ & $1.216\pm0.164$ & $8.879\pm3.113$ & $0.719\pm0.453$ & \\
	& MRS & $12.087\pm1.518$ & $9.852\pm1.265$ & $2.414\pm0.629$ & $22.72\pm1.531$ & $4.879\pm3.482$ & \\
	& Soft-MRS-Exponential & $2.571\pm1.55$ & $2.752\pm1.829$ & $0.442\pm0.172$ & $3.258\pm2.35$ & $0.56\pm0.402$ & \\




In [80]:
result_df["Rank MMD"] = result_df.groupby("Data Set")["MMD Mean"].rank(ascending=True)
result_df["Rank Relative Bias"] = result_df.groupby("Data Set")["Relative Bias Mean"].rank(ascending=True)
result_df[["Method", "Rank MMD", "Rank Relative Bias"]].groupby("Method").mean()

,Rank MMD,Rank Relative Bias
Method,,
KMM,1.8,2.2
MRS,4.0,4.0
PSA,3.0,2.8
Soft-MRS-Exponential,1.2,1.0
Uniform,5.0,5.0
